# U2Flow+FF Flow Inference on Cityscapes Triplets

This notebook runs the KITTI pretrained U2Flow+FF model on 3-frame Cityscapes snippets and produces:
- forward flow (current -> next)
- backward flow from the previous pair
- FF fused flow
- uncertainty and fused mask visualizations
- per-sample output files with flow maps


In [1]:
from pathlib import Path
from typing import Dict, List, Tuple
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

# --- Paths and config ---
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
FLOW_ROOT = PROJECT_ROOT / 'optical_flow_models' / 'post_smurf' / 'U2FLOW'
CHECKPOINT_PATH = FLOW_ROOT / 'checkpoints' / 'kitti.pth'
CITYSCAPES_ROOT = Path('/Users/qbit-glitch/Desktop/datasets/cityscapes')
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks' / 'u2flow_cityscapes_ff_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SPLIT = 'val'
MODEL_INPUT_SIZE = (256, 832)
DEFAULT_SAMPLE_IDS = [
    'frankfurt_000000_000576',
    'frankfurt_000000_001016',
    'frankfurt_000000_001236',
    'frankfurt_000000_001751',
]
MAX_TRIPLETS = 4
FUSION_ITERS = 250
FUSION_LR = 0.01
FUSION_THR = 35.0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import sys
sys.path.insert(0, str(FLOW_ROOT))
sys.path.insert(0, str(FLOW_ROOT / 'core'))

import importlib.util

try:
    spec = importlib.util.spec_from_file_location('cfg_kitti_mv', str(FLOW_ROOT / 'configs' / 'KITTI_MV.py'))
    if spec is None or spec.loader is None:
        raise ImportError('Could not load KITTI_MV config module')
    cfg_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(cfg_module)
except ModuleNotFoundError as e:
    raise ImportError(
        f'Missing dependency while importing U2Flow config: {e}.\n'
        'Install with: pip install -r optical_flow_models/post_smurf/U2FLOW/requirements.txt'
    ) from e

from core.Networks import build_network
from core.utils.warp_utils import get_occu_mask_bidirection
from core.utils.flow_utils import flow_to_image, resize_flow


class TinyModel(nn.Module):
    """Tiny fusion model used by FF pass."""

    def __init__(self):
        super().__init__()
        self._layers = nn.Sequential(
            nn.Conv2d(4, 16, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 2, kernel_size=3, padding=1, bias=True),
        )
        for m in self._layers.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self._layers(x * (-1.0))


COORDS_CACHE: Dict[tuple, torch.Tensor] = {}

def _positions_center_origin(height: int, width: int, device: torch.device) -> torch.Tensor:
    h = torch.linspace(0.0, height - 1, steps=height, dtype=torch.float32, device=device)
    h = h / (height - 1) * 2 - 1
    w = torch.linspace(0.0, width - 1, steps=width, dtype=torch.float32, device=device)
    w = w / (width - 1) * 2 - 1
    h_grid, w_grid = torch.meshgrid(h, w, indexing='ij')
    return torch.stack([h_grid, w_grid], dim=-1)


def train_and_run_tiny_model(flow_forward: torch.Tensor,
                            flow_backward: torch.Tensor,
                            mask_forward: torch.Tensor,
                            mask_backward: torch.Tensor,
                            uncertainty_forward: torch.Tensor,
                            uncertainty_backward: torch.Tensor,
                            iterations: int = FUSION_ITERS,
                            lr: float = FUSION_LR,
                            thr: float = FUSION_THR,
                            device: torch.device = DEVICE):
    batch_size, _, height, width = flow_backward.shape
    key = (height, width, str(device))
    if key not in COORDS_CACHE:
        COORDS_CACHE[key] = _positions_center_origin(height, width, device=device).unsqueeze(0).permute(0, 3, 1, 2)

    coords = COORDS_CACHE[key].repeat(batch_size, 1, 1, 1)
    net_input = torch.cat([flow_backward, coords], dim=1)

    sigma2_f = uncertainty_forward.exp().clamp(min=1e-3, max=200)
    sigma2_b = uncertainty_backward.exp().clamp(min=1e-3, max=200)
    valid_mask = ((sigma2_f < thr) & (sigma2_b < thr)).float()
    if valid_mask.sum() < 1:
        return flow_forward, mask_forward

    model = TinyModel().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.8)

    model.train()
    for step in range(iterations):
        opt.zero_grad()
        pred = model(net_input)
        error = torch.sqrt(torch.sum((pred - flow_forward) ** 2, dim=1, keepdim=True))
        loss = (error * valid_mask).sum() / (valid_mask.sum() + 1e-6)
        loss.backward()
        opt.step()
        if (step + 1) % max(1, iterations // 20) == 0:
            sch.step()

    model.eval()
    with torch.no_grad():
        predicted = model(net_input)

    predicted = torch.nan_to_num(predicted, nan=0.0)
    mask_backward_no_forward = (1 - mask_forward) * mask_backward
    nan_mask = torch.isnan(predicted).any(dim=1, keepdim=True).float()
    mask_backward_no_forward = mask_backward_no_forward * (1 - nan_mask)

    fused_flow = flow_forward * (1 - mask_backward_no_forward) + predicted * mask_backward_no_forward
    fused_mask = torch.clamp(mask_forward + mask_backward, min=0, max=1)
    return fused_flow, fused_mask


def flow_rgb(flow: torch.Tensor) -> np.ndarray:
    arr = flow.detach().cpu().numpy()
    if arr.ndim == 3:
        arr = np.transpose(arr, (1, 2, 0))
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return flow_to_image(arr)

In [2]:
def load_state_dict(path: Path, device: torch.device) -> Dict[str, torch.Tensor]:
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
        ckpt = ckpt['state_dict']
    if not isinstance(ckpt, dict):
        raise TypeError(f'Unsupported checkpoint format: {type(ckpt)}')

    state = {}
    for k, v in ckpt.items():
        if isinstance(k, str) and k.startswith('module.'):
            k = k[7:]
        state[k] = v
    return state


def build_model(device: torch.device) -> nn.Module:
    cfg = cfg_module.get_cfg()
    cfg.mixed_precision = True
    cfg[cfg.network].mixed_precision = True
    model = build_network(cfg).to(device)
    state = load_state_dict(CHECKPOINT_PATH, device)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f'Missing keys: {len(missing)}, unexpected keys: {len(unexpected)}')
    if missing:
        print(list(missing)[:10])
    if unexpected:
        print(list(unexpected)[:10])
    model.eval()
    return model


def load_triplet(paths: List[Path]) -> Tuple[torch.Tensor, Tuple[int, int]]:
    imgs = []
    for p in paths:
        img = np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
        imgs.append(img)
    h, w, _ = imgs[0].shape
    raw = torch.from_numpy(np.stack([np.transpose(im, (2, 0, 1)) for im in imgs], axis=0)).float()
    resized = F.interpolate(raw, size=MODEL_INPUT_SIZE, mode='bilinear', align_corners=False)
    return resized, (h, w)


def parse_cityscapes_triplet_id(sample_id: str, split: str = SPLIT) -> List[Path]:
    """Return a 3-frame triplet from Cityscapes with nearest temporal neighbors."""
    m = re.match(r'^(?P<city>.+?)_(?P<seq>\d{6})_(?P<frame>\d{6})$', sample_id)
    if m is None:
        raise ValueError(f'Invalid sample id: {sample_id}')
    city, seq, frame = m.group('city'), m.group('seq'), int(m.group('frame'))
    city_dir = CITYSCAPES_ROOT / 'leftImg8bit' / split / city
    if not city_dir.exists():
        raise FileNotFoundError(f'City not found: {city_dir}')

    seq_pattern = re.compile(
        rf'^{re.escape(city)}_{re.escape(seq)}_(\d{{6}})_leftImg8bit\.png$'
    )
    seq_frames = []
    for p in sorted(city_dir.glob(f'{city}_{seq}_*_leftImg8bit.png')):
        pm = seq_pattern.match(p.name)
        if pm:
            seq_frames.append((int(pm.group(1)), p))

    if not seq_frames:
        raise RuntimeError(f'No frames found for sequence: {sample_id}')

    seq_frames.sort(key=lambda x: x[0])
    frames = [f for f, _ in seq_frames]
    if frame not in frames:
        raise KeyError(f'Frame {frame} not available in sequence {city}_{seq}')

    idx = frames.index(frame)
    if idx <= 0 or idx >= len(seq_frames) - 1:
        raise RuntimeError(f'No full neighbors for {sample_id}; sequence position must have both prev and next frames')

    return [seq_frames[idx - 1][1], seq_frames[idx][1], seq_frames[idx + 1][1]]


def find_triplet_candidates(split: str = SPLIT, max_triplets: int = MAX_TRIPLETS):
    leftimg_dir = CITYSCAPES_ROOT / 'leftImg8bit' / split
    if not leftimg_dir.exists():
        raise FileNotFoundError(f'Cityscapes split not found: {leftimg_dir}')

    out = []
    for city_dir in sorted(leftimg_dir.glob('*')):
        if not city_dir.is_dir():
            continue
        city = city_dir.name

        seq_map = {}
        city_prefix = re.escape(city)
        for p in city_dir.glob(f'{city}_*_leftImg8bit.png'):
            pm = re.match(rf'^{city_prefix}_(\d{{6}})_(\d{{6}})_leftImg8bit\.png$', p.name)
            if not pm:
                continue
            seq, frame = pm.group(1), int(pm.group(2))
            seq_map.setdefault(seq, []).append((frame, p))

        for seq, items in seq_map.items():
            items.sort(key=lambda x: x[0])
            for i in range(1, len(items) - 1):
                frame = items[i][0]
                sid = f'{city}_{seq}_{frame:06d}'
                out.append((sid, [items[i - 1][1], items[i][1], items[i + 1][1]]))
                if len(out) >= max_triplets:
                    return out

    if not out:
        raise RuntimeError('No triplets found')
    return out

def run_u2flow_ff_on_triplet(model: nn.Module, paths: List[Path], device: torch.device):
    triplet_tensor, (h, w) = load_triplet(paths)
    triplet = triplet_tensor.to(device).unsqueeze(0)

    with torch.no_grad():
        out_fwd = model(triplet[:, 1:, ...])
    flow12_f = out_fwd['flows_f12'][-1]
    flow21_b = out_fwd['flows_b21'][-1]
    unc_f = out_fwd['uncertainty_f12'][-1]

    with torch.no_grad():
        out_bwd = model(triplet[:, :-1, ...])
    flow01_f = out_bwd['flows_f12'][-1]
    flow10_b = out_bwd['flows_b21'][-1]
    unc_b = out_bwd['uncertainty_b21'][-1]

    mask_f = 1 - get_occu_mask_bidirection(flow12_f, flow21_b)
    mask_b = 1 - get_occu_mask_bidirection(flow10_b, flow01_f)

    fused_flow, fused_mask = train_and_run_tiny_model(
        flow_forward=flow12_f,
        flow_backward=flow01_f,
        mask_forward=mask_f,
        mask_backward=mask_b,
        uncertainty_forward=unc_f,
        uncertainty_backward=unc_b,
        iterations=FUSION_ITERS,
        lr=FUSION_LR,
        thr=FUSION_THR,
        device=device,
    )

    up_flow = lambda x: resize_flow(x, (h, w))[0].cpu().numpy()
    up_scalar = lambda x: F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)[0].cpu().numpy()
    return {
        'flow_fwd_up': np.transpose(up_flow(flow12_f), (1, 2, 0)),
        'flow_bwd_up': np.transpose(up_flow(flow01_f), (1, 2, 0)),
        'flow_fused_up': np.transpose(up_flow(fused_flow), (1, 2, 0)),
        'unc_f_up': up_scalar(unc_f)[0],
        'unc_b_up': up_scalar(unc_b)[0],
        'vis_mask_f_up': F.interpolate(mask_f, size=(h, w), mode='nearest')[0, 0].cpu().numpy(),
        'vis_mask_b_up': F.interpolate(mask_b, size=(h, w), mode='nearest')[0, 0].cpu().numpy(),
        'fused_mask_up': F.interpolate(fused_mask, size=(h, w), mode='nearest')[0, 0].cpu().numpy(),
        'sampled_paths': [str(p) for p in paths],
    }

In [3]:
def plot_triplet_result(result: Dict[str, np.ndarray], sample_id: str, save_to: Path | None = None):
    imgs = [np.array(Image.open(Path(p)).convert('RGB')) for p in result['sampled_paths']]

    fig, axs = plt.subplots(3, 3, figsize=(17, 14))
    titles = ['prev', 'current', 'next']
    for i, img in enumerate(imgs):
        axs[0, i].imshow(img)
        axs[0, i].set_title(f'input: {titles[i]}')
        axs[0, i].axis('off')

    axs[1, 0].imshow(flow_rgb(torch.from_numpy(np.transpose(result['flow_fwd_up'], (2, 0, 1)))))
    axs[1, 0].set_title('forward cur->next')
    axs[1, 0].axis('off')

    axs[1, 1].imshow(flow_rgb(torch.from_numpy(np.transpose(result['flow_bwd_up'], (2, 0, 1)))))
    axs[1, 1].set_title('flow prev->cur')
    axs[1, 1].axis('off')

    axs[1, 2].imshow(flow_rgb(torch.from_numpy(np.transpose(result['flow_fused_up'], (2, 0, 1)))))
    axs[1, 2].set_title('fused flow')
    axs[1, 2].axis('off')

    for ax, key, title, cmap in [
        (axs[2, 0], 'unc_f_up', 'log uncertainty forward', 'magma'),
        (axs[2, 1], 'unc_b_up', 'log uncertainty backward', 'magma'),
        (axs[2, 2], 'fused_mask_up', 'fused mask', 'viridis'),
    ]:
        im = result[key]
        if 'unc' in key:
            im = np.log1p(im)
            lo, hi = np.percentile(im, [5, 95])
            implot = ax.imshow(im, cmap=cmap, vmin=lo, vmax=hi)
        else:
            implot = ax.imshow(im, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')
        fig.colorbar(implot, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    fig.suptitle(f'U2Flow+FF - {sample_id}', y=0.99)
    if save_to is not None:
        fig.savefig(save_to, dpi=180)
    return fig


def run_all():
    model = build_model(DEVICE)

    selected = []
    for sid in DEFAULT_SAMPLE_IDS:
        try:
            selected.append((sid, parse_cityscapes_triplet_id(sid)))
        except Exception as e:
            print(f'Skipping {sid}: {e}')

    if not selected:
        selected = find_triplet_candidates(max_triplets=MAX_TRIPLETS)
    else:
        selected = selected[:MAX_TRIPLETS]

    outputs = []
    for sid, triplet_paths in selected:
        try:
            out = run_u2flow_ff_on_triplet(model, triplet_paths, DEVICE)
            out_dir = OUTPUT_ROOT / sid
            out_dir.mkdir(parents=True, exist_ok=True)
            np.savez(
                out_dir / f'{sid}_flows.npz',
                flow_fwd=out['flow_fwd_up'],
                flow_bwd=out['flow_bwd_up'],
                flow_fused=out['flow_fused_up'],
                unc_f=out['unc_f_up'],
                unc_b=out['unc_b_up'],
                vis_mask_f=out['vis_mask_f_up'],
                vis_mask_b=out['vis_mask_b_up'],
                fused_mask=out['fused_mask_up'],
                sampled_paths=np.array(out['sampled_paths'], dtype='<U200'),
            )
            fig_path = out_dir / f'{sid}_viz.png'
            fig = plot_triplet_result(out, sid, save_to=fig_path)
            outputs.append((sid, out_dir, fig_path, out))
            print(f'Saved outputs for {sid} -> {out_dir}')
            plt.close(fig)
        except Exception as e:
            print(f'Failed {sid}: {e}')
    return outputs


In [4]:
# Execute
results = run_all()
len(results)
print('Completed:', len(results))
for r in results:
    print(r[0], '->', r[1])


Missing keys: 0, unexpected keys: 0


/Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/optical_flow_models/post_smurf/U2FLOW/core/Networks/raft_2f_u.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/Users/qbit-glitch/Desktop/datasets/.venv_py310/lib/python3.10/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
/Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/optical_flow_models/post_smurf/U2FLOW/core/Networks/raft_2f_u.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/optical_flow_models/post_smurf/U2FLOW/core/Networks/raft_2f_u.py:104: FutureWarning: `to

Saved outputs for frankfurt_000000_000576 -> /Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/notebooks/u2flow_cityscapes_ff_outputs/frankfurt_000000_000576
Saved outputs for frankfurt_000000_001016 -> /Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/notebooks/u2flow_cityscapes_ff_outputs/frankfurt_000000_001016
Saved outputs for frankfurt_000000_001236 -> /Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/notebooks/u2flow_cityscapes_ff_outputs/frankfurt_000000_001236
Saved outputs for frankfurt_000000_001751 -> /Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/notebooks/u2flow_cityscapes_ff_outputs/frankfurt_000000_001751
Completed: 4
frankfurt_000000_000576 -> /Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/notebooks/u2flow_cityscapes_ff_outputs/frankfurt_000000_000576
frankfurt_000000_001016 -> /Users/qbit-glitch/Desktop/coding-projects/mbps_panoptic_segmentation/notebooks/u2flow_c

## Quick controls
- Set `FUSION_ITERS` to 2000 for the exact script behavior (slow).
- Set `DEFAULT_SAMPLE_IDS` to your preferred Cityscapes frame ids.
- Output directory:
  - `notebooks/u2flow_cityscapes_ff_outputs/<sample_id>/`
  - `*_flows.npz` for raw arrays
  - `*_viz.png` for flow and mask figures
